# B08 · WASB fine-tuned on our hand clicks (Google Colab)

**hoop-reid-tracker**, ball chapter, step B08.

### Why
B07: WASB (Tarashima et al., BMVC 2023, code MIT, https://github.com/nttcom/WASB-SBDT), with its NBA basketball weights
and no training, found 40 % of the balls on our test clips with **100 % precision** (our YOLO detector + tracker:
78 % / 73 %). It sees 3 frames at once and predicts a heatmap, so still look-alikes do not fool it, but it misses most
balls outside the NBA style (FIBA clips: 20-34 % found). B08 starts from those weights and fine-tunes them on our
1,245 hand clicks of the SportsMOT **train** clips (other leagues, SportsMOT's resolution). The test clips are never
used here; the model is scored on them on the laptop, exactly as B04-B07.

### Data (`scripts/ball/b08_export_wasb_data.py` -> `wasb_ft.zip` in `MyDrive/hooptrack/`)
Every clicked frame with its two neighbours on each side, at WASB's input size (512 x 288). A clicked frame `f` gives
3 windows (f-2..f, f-1..f+1, f..f+2); only the output of frame `f` has a target (the click, or empty if the ball is
not visible); the other two outputs are left out of the loss. The last clip of each train game is the val split.

### Training
WASB's own HRNet and target (a disk of radius 2.5 px, values 1 -> 0.6), its quality focal loss, Adam 1e-4,
horizontal flips and mild colour changes. Best epoch by F1 on the val clicks (found = blob within 5 px at 512 x 288,
WASB's own tolerance is 4).

### Steps
1. Upload `data/wasb_ft.zip` to `MyDrive/hooptrack/`. 2. Runtime -> T4 GPU -> Run all.
3. Download `MyDrive/hooptrack/runs/wasb_ft_b08/best.pth.tar` to `weights/wasb_ft_b08.pth.tar` on the laptop.


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip -q install gdown pyyaml
!test -d WASB-SBDT || git clone -q https://github.com/nttcom/WASB-SBDT

In [ ]:
import os, sys, zipfile, math, random, time, json
from pathlib import Path
import numpy as np, pandas as pd, cv2, yaml, torch

IN_COLAB = "google.colab" in str(get_ipython()) if "get_ipython" in globals() else False
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_DIR, WORK = Path("/content/drive/MyDrive/hooptrack"), Path("/content")
else:
    DRIVE_DIR, WORK = Path(os.environ.get("HOOPTRACK_DIR", "data")), Path(os.environ.get("HOOPTRACK_WORK", "/tmp/hooptrack"))
ZIP = DRIVE_DIR / "wasb_ft.zip"
RUN = DRIVE_DIR / "runs" / "wasb_ft_b08"; RUN.mkdir(parents=True, exist_ok=True)
WASB_BASKETBALL_ID = "1nfECuSyJvPUmz3njZCdFERSQQbERt8FU"      # WASB model zoo: basketball weights
EPOCHS, LR, BATCH, SEED = 30, 1e-4, 8, 0
W, H, R, MIN_V = 512, 288, 2.5, 0.6                           # input size; target disk radius and edge value (WASB)
TOL = 5.0                                                    # px at 512 x 288 for a "found" ball
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print(DEVICE, RUN)

## Data

In [ ]:
DATA = WORK / "wasb_ft"
if not (DATA / "labels.csv").is_file():
    with zipfile.ZipFile(ZIP) as z:
        z.extractall(DATA)
labels = pd.read_csv(DATA / "labels.csv")
frames = {c: {int(p.stem) for p in (DATA / "frames" / c).glob("*.jpg")} for c in labels["clip"].unique()}

def windows(split):
    """(clip, first frame, index of the labelled frame in the window, x, y, visible) for every usable window."""
    out = []
    for r in labels[labels["split"] == split].itertuples():
        for o in ((0, 1, 2) if split == "train" else (1,)):
            first = r.frame - o
            if all(first + k in frames[r.clip] for k in range(3)):
                out.append((r.clip, first, o, r.x, r.y, bool(r.visible)))
    return out

TRAIN, VAL = windows("train"), windows("val")
print(len(TRAIN), "train windows,", len(VAL), "val windows")

In [ ]:
MEAN, STD = np.array([0.485, 0.456, 0.406], np.float32), np.array([0.229, 0.224, 0.225], np.float32)
YY, XX = np.mgrid[0:H, 0:W]

def target(x, y, visible):
    """WASB's binary_fixed_size target: 1 at the centre down to MIN_V at radius R, 0 outside."""
    if not visible:
        return np.zeros((H, W), np.float32)
    d = np.hypot(XX - x, YY - y)
    return np.where(d <= R, 1 - (1 - MIN_V) * d / R, 0).astype(np.float32)

class Windows(torch.utils.data.Dataset):
    def __init__(self, items, train):
        self.items, self.train = items, train
    def __len__(self):
        return len(self.items)
    def __getitem__(self, i):
        clip, first, o, x, y, vis = self.items[i]
        imgs = [cv2.cvtColor(cv2.imread(str(DATA / "frames" / clip / f"{first + k:06d}.jpg")), cv2.COLOR_BGR2RGB)
                .astype(np.float32) / 255 for k in range(3)]
        if self.train:
            if random.random() < 0.5:                                     # horizontal flip
                imgs = [im[:, ::-1] for im in imgs]
                x = W - 1 - x if vis else x
            g = random.uniform(0.85, 1.15); b = random.uniform(-0.05, 0.05)  # mild brightness / contrast
            imgs = [np.clip(im * g + b, 0, 1) for im in imgs]
        x_in = np.concatenate([((im - MEAN) / STD).transpose(2, 0, 1) for im in imgs], 0)
        return torch.from_numpy(np.ascontiguousarray(x_in)), torch.from_numpy(target(x, y, vis)), o

train_loader = torch.utils.data.DataLoader(Windows(TRAIN, True), batch_size=BATCH, shuffle=True, num_workers=2, drop_last=True)
val_loader = torch.utils.data.DataLoader(Windows(VAL, False), batch_size=BATCH, shuffle=False, num_workers=2)

## WASB's network with its basketball weights

In [ ]:
sys.path.insert(0, "WASB-SBDT/src")
from models.hrnet import HRNet

class AttrDict(dict):
    def __init__(self, d):
        super().__init__({k: AttrDict(v) if isinstance(v, dict) else v for k, v in d.items()})
    def __getattr__(self, k):
        try:
            return self[k]
        except KeyError as e:
            raise AttributeError(k) from e

cfg = AttrDict(yaml.safe_load(open("WASB-SBDT/src/configs/model/wasb.yaml")))
model = HRNet(cfg)
w0 = WORK / "wasb_basketball_best.pth.tar"
if not w0.is_file():
    import gdown
    gdown.download(id=WASB_BASKETBALL_ID, output=str(w0), quiet=False)
state = torch.load(w0, map_location="cpu", weights_only=False)
state = state.get("model_state_dict", state)
model.load_state_dict({k.removeprefix("module."): v for k, v in state.items()})
model = model.to(DEVICE)
print(sum(p.numel() for p in model.parameters()) / 1e6, "M parameters")

## Loss, evaluation

In [ ]:
def qfl(logits, y, beta=2.0):
    """Quality focal loss on soft targets (WASB's loss): (|y - p|^beta) * BCE."""
    p = torch.sigmoid(logits)
    bce = torch.nn.functional.binary_cross_entropy_with_logits(logits, y, reduction="none")
    return ((y - p).abs() ** beta * bce).sum() / max(1.0, (y > 0).sum().item())

def blobs(hm, th=0.5):
    if hm.max() <= th:
        return []
    n, lab = cv2.connectedComponents((hm > th).astype(np.uint8))
    out = []
    for m in range(1, n):
        ys, xs = np.nonzero(lab == m); w = hm[ys, xs]
        out.append(((xs * w).sum() / w.sum(), (ys * w).sum() / w.sum(), w.sum()))
    return out

@torch.inference_mode()
def evaluate():
    model.eval(); found = answers = visible = 0
    for x, y, o in val_loader:
        hm = torch.sigmoid(model(x.to(DEVICE))[0]).cpu().numpy()
        for i in range(len(x)):
            h = hm[i, o[i]]; t = y[i].numpy(); vis = t.max() > 0
            b = blobs(h)
            visible += int(vis)
            if b:
                answers += 1
                bx, by, _ = max(b, key=lambda z: z[2])
                if vis:
                    cy, cx = np.unravel_index(t.argmax(), t.shape)
                    found += int(np.hypot(bx - cx, by - cy) <= TOL)
    r, p = found / max(1, visible), found / max(1, answers)
    return {"recall": round(r, 3), "precision": round(p, 3), "f1": round(2 * r * p / (r + p), 3) if r + p else 0.0}

before = evaluate(); print("WASB basketball weights, before fine-tuning (val clicks):", before)

## Fine-tuning

In [ ]:
opt = torch.optim.Adam(model.parameters(), lr=LR)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, EPOCHS * len(train_loader))
scaler = torch.amp.GradScaler(enabled=DEVICE == "cuda")
history, best = [{"epoch": 0, **before}], before["f1"]
torch.save({"model_state_dict": model.state_dict(), "epoch": 0, "val": before}, RUN / "best.pth.tar")
for epoch in range(1, EPOCHS + 1):
    model.train(); t0 = time.time(); tot = 0.0
    for x, y, o in train_loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        with torch.autocast(DEVICE, enabled=DEVICE == "cuda"):
            out = model(x)[0]
        sel = out.float()[torch.arange(len(o)), o.to(DEVICE)]          # only the labelled frame of each window
        loss = qfl(sel, y)
        opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
        tot += loss.item()
    v = evaluate(); history.append({"epoch": epoch, "loss": round(tot / len(train_loader), 4), **v})
    print(history[-1], f"{time.time() - t0:.0f}s")
    if v["f1"] > best:
        best = v["f1"]
        torch.save({"model_state_dict": model.state_dict(), "epoch": epoch, "val": v}, RUN / "best.pth.tar")
(RUN / "history.json").write_text(json.dumps(history, indent=2))
print("best val F1", best, "->", RUN / "best.pth.tar")